# Experimento 1 — Rendimiento de I/O: HDFS vs. Sistema Local
**Procesamiento de Alto Volúmen de Datos — PUJ**

## Objetivo
Medir y comparar el **throughput de lectura/escritura** al usar HDFS distribuido frente al sistema de ficheros local, para diferentes tamaños de archivos CSV sintéticos.

## Hipótesis
HDFS supera al almacenamiento local en lectura de archivos grandes gracias al paralelismo de sus DataNodes, pero tiene mayor latencia en archivos pequeños por la sobrecarga del NameNode.

## Métricas
- Tiempo de escritura (s)
- Tiempo de lectura (s)  
- Throughput (MB/s)
- Número de bloques HDFS generados

In [1]:
# ── Configuración ──────────────────────────────────────────────────────────
import os, time, subprocess, random, string
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pyspark.sql import SparkSession
from pyspark.sql.types import *

MASTER_IP    = '10.43.97.145'        # ← Cambia por tu IP
SPARK_MASTER = f'spark://{MASTER_IP}:7077'
HDFS_URI     = f'hdfs://{MASTER_IP}:9000'
CONDA_ENV    = '/opt/conda_envs/pyspark_env'
HADOOP_HOME     = '/nfs/condor/apps/hadoop-3.5.0'
HADOOP_BIN   = f'{HADOOP_HOME}/bin/hdfs'
LOCAL_DIR    = '/tmp/exp1_local'

os.makedirs(LOCAL_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('EXP1_IO_HDFS_vs_Local')
    .master(SPARK_MASTER)
    .config('spark.driver.host', '10.43.97.145')
    .config('spark.driver.bindAddress', '10.43.97.145')
    .config('spark.pyspark.python', f'{CONDA_ENV}/bin/python')
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory', '2g')
    .config('spark.hadoop.fs.defaultFS', HDFS_URI)
    .getOrCreate()
)
print('SparkSession lista')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SparkSession lista


## 1.1 Generar datasets sintéticos de distintos tamaños

In [2]:
def generar_csv(ruta_local: str, num_filas: int):
    """Genera un CSV con datos numéricos y de texto aleatorio."""
    rng = random.Random(42)
    with open(ruta_local, 'w') as f:
        f.write('id,nombre,edad,salario,departamento,score\n')
        departamentos = ['Ingenieria','Finanzas','Marketing','RRHH','IT','Legal']
        for i in range(num_filas):
            nombre  = ''.join(rng.choices(string.ascii_lowercase, k=8))
            edad    = rng.randint(22, 65)
            salario = round(rng.uniform(30000, 150000), 2)
            depto   = rng.choice(departamentos)
            score   = round(rng.uniform(0, 100), 4)
            f.write(f'{i},{nombre},{edad},{salario},{depto},{score}\n')
    tamanio_mb = os.path.getsize(ruta_local) / (1024**2)
    print(f'{ruta_local}  →  {num_filas:,} filas  |  {tamanio_mb:.1f} MB')
    return tamanio_mb

# Tamaños: ~1 MB, ~50 MB, ~200 MB
datasets = [
    {'label': '1 MB',   'filas': 13_000},
    {'label': '50 MB',  'filas': 650_000},
    {'label': '200 MB', 'filas': 2_600_000},
]

print('Generando archivos locales...')
for ds in datasets:
    path = f"{LOCAL_DIR}/datos_{ds['label'].replace(' ','')}.csv"
    ds['local_path'] = path
    ds['size_mb']    = generar_csv(path, ds['filas'])
print('\nDatasets listos.')

Generando archivos locales...
/tmp/exp1_local/datos_1MB.csv  →  13,000 filas  |  0.5 MB
/tmp/exp1_local/datos_50MB.csv  →  650,000 filas  |  26.8 MB
/tmp/exp1_local/datos_200MB.csv  →  2,600,000 filas  |  109.1 MB

Datasets listos.


## 1.2 Medir escritura: local → HDFS (`hdfs dfs -put`)

In [3]:
def hdfs_put(local_path: str, hdfs_dir: str):
    """Copia un archivo local a HDFS y mide el tiempo."""
    t0 = time.perf_counter()
    subprocess.run([HADOOP_BIN, 'dfs', '-put', '-f', local_path, hdfs_dir], check=True)
    return time.perf_counter() - t0

print('Midiendo escritura a HDFS...\n')
for ds in datasets:
    hdfs_path = f"/experimentos/exp1_io/datos_{ds['label'].replace(' ','')}.csv"
    ds['hdfs_path']      = hdfs_path
    ds['tiempo_put']     = hdfs_put(ds['local_path'], '/experimentos/exp1_io/')
    ds['throughput_put'] = ds['size_mb'] / ds['tiempo_put']
    print(f"  {ds['label']:8s} → {ds['tiempo_put']:.2f}s  |  {ds['throughput_put']:.1f} MB/s")

Midiendo escritura a HDFS...



2026-05-21 19:49:54,397 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:54,430 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:55,008 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:55,031 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:55,229 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:55,256 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:55,331 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19

  1 MB     → 2.81s  |  0.2 MB/s


2026-05-21 19:49:57,124 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:57,151 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:57,746 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:57,768 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:57,945 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:57,960 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:58,032 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19

  50 MB    → 2.82s  |  9.5 MB/s


2026-05-21 19:49:59,921 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:49:59,953 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:50:00,488 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:50:00,509 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:50:00,714 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:50:00,737 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:50:00,804 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19

  200 MB   → 3.13s  |  34.8 MB/s


## 1.3 Verificar bloques replicados en HDFS

In [4]:
for ds in datasets:
    result = subprocess.run(
        [HADOOP_BIN, 'fsck', ds['hdfs_path'], '-files', '-blocks'],
        capture_output=True, text=True
    )
    # Extraer número de bloques de la salida
    for line in result.stdout.splitlines():
        if 'Total blocks' in line or 'blks' in line.lower():
            ds['bloques'] = line.strip()
            break
    else:
        ds['bloques'] = 'ver salida completa'
    print(f"  {ds['label']:8s} → {ds['bloques']}")
    print(result.stdout[-300:])   # Últimas líneas del fsck
    print('─' * 60)

  1 MB     → Total blocks (validated):	1 (avg. block size 540316 B)
block groups:	0
 Average block group size:	0.0
 Missing block groups:		0
 Corrupt block groups:		0
 Missing internal blocks:	0
 Blocks queued for replication:	0
FSCK ended at Thu May 21 19:50:26 COT 2026 in 63 milliseconds


The filesystem under path '/experimentos/exp1_io/datos_1MB.csv' is HEALTHY

────────────────────────────────────────────────────────────
  50 MB    → Total blocks (validated):	1 (avg. block size 28126493 B)
block groups:	0
 Average block group size:	0.0
 Missing block groups:		0
 Corrupt block groups:		0
 Missing internal blocks:	0
 Blocks queued for replication:	0
FSCK ended at Thu May 21 19:50:28 COT 2026 in 6 milliseconds


The filesystem under path '/experimentos/exp1_io/datos_50MB.csv' is HEALTHY

────────────────────────────────────────────────────────────
  200 MB   → Total blocks (validated):	1 (avg. block size 114433579 B)
lock groups:	0
 Average block group size:	0.0
 Missing block groups

## 1.4 Medir lectura Spark desde HDFS vs. sistema local

In [6]:
def medir_lectura_spark(path: str, es_hdfs: bool = True) -> tuple:
    """Lee un CSV con Spark y devuelve (tiempo_s, num_filas)."""
    full_path = path if es_hdfs else f'file://{path}'
    t0 = time.perf_counter()
    df = spark.read.csv(full_path, header=True, inferSchema=True)
    n  = df.count()        # acción que fuerza la lectura real
    t1 = time.perf_counter()
    return t1 - t0, n

print('Midiendo lectura HDFS vs. Local con Spark...\n')
for ds in datasets:
    # Lectura HDFS
    hdfs_full = f"{HDFS_URI}{ds['hdfs_path']}"
    ds['tiempo_read_hdfs'], filas = medir_lectura_spark(hdfs_full)
    ds['throughput_read_hdfs'] = ds['size_mb'] / ds['tiempo_read_hdfs']

    # Lectura local
    ds['tiempo_read_local'], _ = medir_lectura_spark(ds['local_path'], es_hdfs=False)
    ds['throughput_read_local'] = ds['size_mb'] / ds['tiempo_read_local']

    print(f"  {ds['label']:8s} | HDFS: {ds['tiempo_read_hdfs']:.2f}s "
          f"({ds['throughput_read_hdfs']:.1f} MB/s) | "
          f"Local: {ds['tiempo_read_local']:.2f}s "
          f"({ds['throughput_read_local']:.1f} MB/s) | Filas: {filas:,}")

Midiendo lectura HDFS vs. Local con Spark...



26/05/21 19:55:43 WARN TaskSetManager: Lost task 0.0 in stage 10.0 (TID 8) (10.43.97.135 executor 0): org.apache.spark.SparkFileNotFoundException: File file:/tmp/exp1_local/datos_1MB.csv does not exist
It is possible the underlying files have been updated. You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.readCurrentFileNotFoundError(QueryExecutionErrors.scala:781)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:220)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:279)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressi

Py4JJavaError: An error occurred while calling o64.csv.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 10.0 failed 4 times, most recent failure: Lost task 0.3 in stage 10.0 (TID 11) (10.43.97.141 executor 3): org.apache.spark.SparkFileNotFoundException: File file:/tmp/exp1_local/datos_1MB.csv does not exist
It is possible the underlying files have been updated. You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.readCurrentFileNotFoundError(QueryExecutionErrors.scala:781)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:220)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:279)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$an

## 1.5 Tabla de resultados y visualización

In [ ]:
# ── Tabla resumen ──────────────────────────────────────────────────────────
resultados = pd.DataFrame([{
    'Tamaño'           : ds['label'],
    'MB reales'        : round(ds['size_mb'], 1),
    'Put HDFS (s)'     : round(ds['tiempo_put'], 2),
    'TP Put (MB/s)'    : round(ds['throughput_put'], 1),
    'Read HDFS (s)'    : round(ds['tiempo_read_hdfs'], 2),
    'TP HDFS (MB/s)'   : round(ds['throughput_read_hdfs'], 1),
    'Read Local (s)'   : round(ds['tiempo_read_local'], 2),
    'TP Local (MB/s)'  : round(ds['throughput_read_local'], 1),
} for ds in datasets])

print('\n📊 TABLA DE RESULTADOS — Experimento 1')
print(resultados.to_string(index=False))
resultados.to_csv('/tmp/exp1_resultados.csv', index=False)
print('\n💾 Guardado en /tmp/exp1_resultados.csv')

In [ ]:
# ── Gráficas ───────────────────────────────────────────────────────────────
labels   = [ds['label'] for ds in datasets]
x        = range(len(labels))
width    = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 1 — I/O HDFS vs. Local', fontsize=14, fontweight='bold')

# Throughput de lectura
ax = axes[0]
b1 = ax.bar([i - width/2 for i in x], [ds['throughput_read_hdfs']  for ds in datasets],
            width, label='HDFS', color='#1F77B4')
b2 = ax.bar([i + width/2 for i in x], [ds['throughput_read_local'] for ds in datasets],
            width, label='Local', color='#FF7F0E')
ax.set_title('Throughput de Lectura (MB/s)')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('MB/s'); ax.legend()
ax.bar_label(b1, fmt='%.1f', padding=2)
ax.bar_label(b2, fmt='%.1f', padding=2)

# Tiempo de lectura
ax2 = axes[1]
b3 = ax2.bar([i - width/2 for i in x], [ds['tiempo_read_hdfs']  for ds in datasets],
             width, label='HDFS', color='#1F77B4')
b4 = ax2.bar([i + width/2 for i in x], [ds['tiempo_read_local'] for ds in datasets],
             width, label='Local', color='#FF7F0E')
ax2.set_title('Tiempo de Lectura (s)')
ax2.set_xticks(list(x)); ax2.set_xticklabels(labels)
ax2.set_ylabel('Segundos'); ax2.legend()
ax2.bar_label(b3, fmt='%.2f', padding=2)
ax2.bar_label(b4, fmt='%.2f', padding=2)

plt.tight_layout()
plt.savefig('/tmp/exp1_grafica.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Gráfica guardada en /tmp/exp1_grafica.png')

## 1.6 Análisis de resultados

> **Completa esta sección con tus observaciones:**
>
> - ¿Para qué tamaño de archivo HDFS supera al almacenamiento local?
> - ¿Cuántos bloques generó HDFS para el archivo de 200 MB? (bloque por defecto = 128 MB)
> - ¿Cómo afecta el factor de replicación (2) al tiempo de escritura vs. lectura?
> - ¿Qué patrón observas en el throughput al crecer el tamaño del archivo?

In [7]:
spark.stop()
print('SparkSession detenida.')

SparkSession detenida.
